# <font color="steelblue">Indicadores de salud y diabetes</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.


## <font color="steelblue">Objetivos del proyecto</font>

Construir, comparar y **desplegar** un sistema de clasificación que estime el **riesgo de diabetes** a partir de indicadores de salud, siguiendo un flujo de trabajo profesional. Al terminar, debéis ser capaces de:

* Plantear correctamente un problema de **clasificación multiclase** con clases **muy desequilibradas**.
* Construir y **comparar** varias familias de clasificadores con una metodología sólida (validación cruzada, métricas adecuadas, sin fugas de datos).
* Tratar el desequilibrio mediante **ponderación de muestras** y **remuestreo**, y medir su efecto.
* **Optimizar los hiperparámetros** de los mejores candidatos.
* **Combinar modelos** (*voting* / *stacking*) cuando aporte mejora.
* **Interpretar** el modelo final y **desplegarlo** como una herramienta utilizable.

## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

Los datos proceden del **BRFSS** (*Behavioral Risk Factor Surveillance System*), la gran **encuesta telefónica de salud** que los CDC realizan anualmente en Estados Unidos. Alex Teboul publicó en Kaggle una versión **limpia** de la edición de 2015, con **253.680 registros** y **21 variables predictoras** más la objetivo. Es esencial retener el origen: no se trata de historias clínicas ni de mediciones de laboratorio, sino de **respuestas autoinformadas** por los encuestados a preguntas concretas. Cada variable es, en el fondo, la codificación de una pregunta de la encuesta.

**Variable objetivo — `Diabetes_012`** (ordinal de 3 clases):

| Valor | Significado |
|---|---|
| `0` | **Sin diabetes** —o diabetes **únicamente durante el embarazo** (diabetes gestacional)—. |
| `1` | **Prediabetes**. |
| `2` | **Diabetes**. |

El reparto está **muy desequilibrado**: la clase sin diabetes domina el conjunto, mientras que la **prediabetes es una clase extremadamente minoritaria** (en torno al 2 % de los registros). Conviene comprobarlo con `value_counts(normalize=True)` antes de decidir la estrategia de evaluación.

### <font color="steelblue">Diccionario de variables</font>

**Bloque 1 — Binarias (0 = no, 1 = sí)**

| Variable | Pregunta o significado |
|---|---|
| `HighBP` | ¿Un profesional sanitario le ha dicho alguna vez que tiene **hipertensión**? |
| `HighChol` | ¿Le han dicho alguna vez que tiene el **colesterol alto**? |
| `CholCheck` | ¿Se ha medido el colesterol en los **últimos 5 años**? Mide acceso y seguimiento sanitario, no salud. |
| `Stroke` | ¿Le han dicho alguna vez que ha sufrido un **ictus**? |
| `HeartDiseaseorAttack` | Antecedente de **cardiopatía coronaria** o **infarto de miocardio**. |
| `Smoker` | ¿Ha fumado al menos **100 cigarrillos** en toda su vida? (unos 5 paquetes). No mide consumo actual. |
| `PhysActivity` | **Actividad física** en los últimos 30 días, sin contar la del trabajo. |
| `Fruits` | Consume **fruta** al menos una vez al día. |
| `Veggies` | Consume **verdura** al menos una vez al día. |
| `HvyAlcoholConsump` | **Consumo elevado de alcohol**: más de 14 bebidas semanales en hombres, más de 7 en mujeres. **El umbral depende del sexo.** |
| `AnyHealthcare` | Dispone de algún tipo de **cobertura sanitaria**. |
| `NoDocbcCost` | En los últimos 12 meses, ¿necesitó ir al médico y **no pudo por el coste**? |
| `DiffWalk` | ¿Tiene **dificultad seria para caminar** o subir escaleras? |
| `Sex` | 0 = mujer, 1 = hombre. |

**Bloque 2 — Ordinales**

| Variable | Escala | Significado |
|---|---|---|
| `GenHlth` | 1–5 | **Salud general autopercibida**. ⚠️ **La escala está invertida:** 1 = excelente, 5 = mala. Un valor alto es *peor* salud. |
| `Age` | 1–13 | **Grupo de edad** en tramos de cinco años: 1 = 18–24 años, y así sucesivamente hasta 13 = 80 años o más. **No es la edad en años.** |
| `Education` | 1–6 | **Nivel educativo**, de 1 (nunca escolarizado) a 6 (estudios universitarios completos). |
| `Income` | 1–8 | **Tramo de ingresos** del hogar, de 1 (menos de 10.000 $ anuales) a 8 (75.000 $ o más). |

**Bloque 3 — Numéricas**

| Variable | Rango | Significado |
|---|---|---|
| `BMI` | continuo | **Índice de masa corporal**. Es la **única variable genuinamente continua** del conjunto. |
| `MentHlth` | 0–30 | Número de **días** de los últimos 30 en que la **salud mental** no fue buena. |
| `PhysHlth` | 0–30 | Número de **días** de los últimos 30 en que la **salud física** no fue buena (enfermedad o lesión). |

### <font color="steelblue">Advertencias metodológicas</font>

1. **Cuidado con el sentido de `GenHlth`.** Todas las variables binarias siguen la convención «1 = presencia del problema», pero en `GenHlth` un valor **alto** significa **peor** salud, mientras que en `Education` o `Income` un valor alto es **favorable**. Al interpretar coeficientes o valores SHAP hay que tener presente el sentido de cada escala; es el error de lectura más frecuente en este conjunto.

2. **`Age`, `Education` e `Income` son ordinales, no continuas.** Están **discretizadas en tramos**, de modo que la distancia entre categorías consecutivas no es constante (el salto de `Income` 7 a 8 no equivale al de 1 a 2). Tratarlas como numéricas es habitual y funciona bien con árboles, pero conviene saber que se está imponiendo una equidistancia que los datos no tienen.

3. **La respuesta es ordinal.** `Sin diabetes < Prediabetes < Diabetes` describe una progresión clínica. Confundir prediabetes con diabetes es menos grave que confundirla con ausencia de enfermedad, y las métricas nominales no lo distinguen: acompáñalas de índices **sensibles al orden** (QWK, MAE ordinal) y examina la matriz de confusión.

4. **El desequilibrio hace inútil la exactitud.** Un modelo que prediga siempre «sin diabetes» supera el 80 % de acierto sin haber aprendido nada, y la clase de **prediabetes** —la más interesante desde el punto de vista preventivo, porque es la única reversible— resulta prácticamente invisible para un clasificador ingenuo. Es aquí donde cobran sentido el reequilibrado (SMOTE, `class_weight`) y las métricas por clase.

5. **Todo es autoinformado.** Las variables no miden hechos, sino **lo que las personas dicen**: están sujetas a sesgo de memoria y de deseabilidad social (se infradeclara el tabaco y el alcohol; se declara más actividad física de la real). Y, lo más importante, la etiqueta también es autoinformada: alguien con **diabetes no diagnosticada** aparece codificado como `0`.

6. **El sesgo de detección contamina la etiqueta.** Ese punto anterior tiene una consecuencia sutil y grave: la probabilidad de estar diagnosticado depende del **acceso al sistema sanitario**, que es justamente lo que miden `AnyHealthcare`, `NoDocbcCost` y `CholCheck`. Un modelo puede aprender a predecir **quién ha sido diagnosticado** en lugar de **quién tiene la enfermedad**. Merece la pena discutirlo al interpretar la importancia de esas tres variables.

7. **Causa y consecuencia se confunden.** `HighBP`, `HighChol`, `DiffWalk` o `BMI` comparten factores de riesgo con la diabetes, y algunos pueden ser **consecuencia** de ella más que causa. El modelo predice bien, pero **ninguna de sus importancias admite lectura causal**: no permite concluir que reducir el IMC reduzca el riesgo.

8. **La encuesta original tiene pesos muestrales.** El BRFSS es una muestra compleja, diseñada para ser representativa **solo cuando se aplican los pesos de muestreo**. La versión limpia de Kaggle los ha descartado, de modo que las prevalencias observadas **no son las de la población estadounidense**.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Partición primero.** Separad *train*/*test* (estratificado por la clase) **antes** de cualquier ajuste. El *test* solo se toca al final.
2. **Nada de fugas de datos.** El escalado, la imputación y el **remuestreo** se ajustan **solo con el *train***, y dentro de validación cruzada deben rehacerse en **cada pliegue**: usad un **`Pipeline`** (de *scikit-learn* o de *imbalanced-learn*).
3. **El equilibrado solo en *train*.** El *test* conserva la proporción real de clases (material 11).
4. **Métricas acordes al desequilibrio.** La *accuracy* engaña: priorizad **exactitud balanceada**, **F1-macro** y el **recall por clase** (especialmente prediabetes y diabetes).
5. **Reproducibilidad.** Fijad `random_state` y documentad versiones.
6. **Honestidad.** Reportad también lo que **no** funcionó. Un análisis crítico vale más que un número alto.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

En Colab, descomentad la primera línea para instalar las dependencias. El bloque de descarga es el que se os ha proporcionado.

In [ ]:
# !pip -q install kagglehub imbalanced-learn gradio optuna scikit-learn shap
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import kagglehub
RNG = 42

In [ ]:
# Paso 1: descargar el dataset
path = kagglehub.dataset_download("alexteboul/diabetes-health-indicators-dataset")
print("Ruta al dataset:", path)

# Paso 2: listar los archivos disponibles
archivos = os.listdir(path)
print("Archivos disponibles:")
for f in archivos:
    print(f"  · {f}")

# Paso 3: Cargar el dataset completo con 3 clases (multiclase)
df = pd.read_csv(os.path.join(path, "diabetes_012_health_indicators_BRFSS2015.csv"))
print(f"Dataset multiclase: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# <font color="steelblue">Fase 1 — Comprensión y exploración (EDA)</font>

**Tareas obligatorias**
1. **Distribución del objetivo.** Calculad la frecuencia (absoluta y relativa) de las 3 clases. ¿Cómo de desequilibrado está? ¿Cuál es la clase más rara?
2. **Tipado de variables.** Construid las listas `bin_cols`, `ord_cols`, `num_cols` (apoyaos en la descripción). Verificad rangos y valores posibles.
3. **Calidad de los datos.** Nulos, duplicados, valores fuera de rango, distribución de `BMI`, `MentHlth`, `PhysHlth`.
4. **Relación con el objetivo.** Tasa de diabetes por nivel de `GenHlth`, `HighBP`, `BMI` (agrupado), `Age`/`Income`/`Education`… Gráficos que lo muestren.
5. **Conclusión.** Tres o cuatro hallazgos que orienten el modelado.

> **Pregunta para responder:** con esta distribución de clases, ¿por qué la *accuracy* sería una métrica engañosa? ¿Qué métricas usaréis y por qué?

# <font color="steelblue">Fase 2 — Preprocesado y partición</font>

**Tareas obligatorias**
1. **Separad `X` e `y`** y haced la **partición estratificada** *train*/*test* (p. ej. 80/20).
2. **Estrategia de preprocesado por tipo de variable** con un `ColumnTransformer`:
   * numéricas → **escalado** (`StandardScaler`/`RobustScaler`) **solo** para modelos que lo necesitan (logística, SVM, kNN);
   * los **árboles/ensembles no requieren escalado**;
   * decidid cómo tratar las ordinales (dejarlas como están suele bastar).
3. Encapsulad todo en **`Pipeline`** para evitar fugas (el escalado se ajusta dentro de la CV).

> **Decisión de diseño:** ¿un único preprocesado para todos los modelos, o uno con escalado y otro sin él? Justificadlo.

# <font color="steelblue">Fase 3 — Modelos base y comparación</font>

**Tareas obligatorias**
1. Entrenad y comparad **al menos 5 familias** vistas en el curso, por ejemplo: **Regresión logística**, **kNN**, **SVM**, **Árbol de decisión**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost), **Naive Bayes**.
2. Comparad con **validación cruzada estratificada** (`StratifiedKFold`) sobre el *train*, con una métrica adecuada al desequilibrio (**`balanced_accuracy`** o **`f1_macro`**).
3. Presentad una **tabla** con media ± desviación de la métrica por modelo y comentadla.

> **Aviso de cómputo:** son 253.680 filas. Para esta comparación podéis trabajar con una **submuestra estratificada** (p. ej. 30–50k) y dejar el ajuste final sobre el total. Documentadlo.

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

El objetivo está **muy desequilibrado** (la prediabetes es minoritaria). Aquí debéis **medir el efecto** de tratarlo (material **11. Equilibrando las muestras**).

**Tareas obligatorias** — comparad, sobre los 2–3 mejores modelos de la Fase 3:
1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'` (o `sample_weight`).
3. **Sobremuestreo:** **SMOTE** (o Borderline-SMOTE / ADASYN).
4. **Submuestreo:** `RandomUnderSampler` o **NearMiss**; y/o un **híbrido** (SMOTEENN/SMOTETomek).

Para cada estrategia, reportad **exactitud balanceada**, **F1-macro** y **recall por clase**, y discutid el compromiso (ganar recall en minoritarias suele costar precisión).

> **Imprescindible (sin fugas):** el remuestreo va **dentro** de un `Pipeline` de *imbalanced-learn*, de modo que se aplica **solo al *train* de cada pliegue**. **Nunca** remuestreéis el *test*.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

**Tareas obligatorias**
1. Tomad los **2–3 mejores** (modelo + estrategia de equilibrado) y **optimizad sus hiperparámetros**.
2. Usad `GridSearchCV`, `RandomizedSearchCV` **u Optuna**, con **CV estratificada** y la **misma métrica** que venís usando.
3. Mantened el remuestreo/preprocesado **dentro** del objeto de búsqueda (sobre el `Pipeline`, no sobre los datos sueltos).
4. Reportad los **mejores hiperparámetros** y la **mejora** frente a los valores por defecto.

> Con tantos datos, `RandomizedSearchCV`/Optuna sobre una **submuestra** suele ser más eficiente que una rejilla exhaustiva. Reentrenad el ganador sobre todo el *train*.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

**Tareas obligatorias**
1. Combinad vuestros mejores modelos ya optimizados con **`VotingClassifier`** (votación **blanda**, promediando probabilidades) y/o **`StackingClassifier`** (un metamodelo, p. ej. regresión logística, sobre las predicciones de los base).
2. Comparad el conjunto frente al **mejor modelo individual**: ¿mejora la métrica? ¿compensa el coste y la pérdida de interpretabilidad?
3. **Decisión justificada:** combinar **solo si aporta** mejora real (recordad el requisito "combinación de modelos *si fuera necesario*").

# <font color="steelblue">Fase 7 — Evaluación final e interpretación</font>

**Tareas obligatorias** (¡ahora sí se usa el *test*, una sola vez!)
1. Evaluad el **modelo final elegido** sobre el *test*: **matriz de confusión**, `classification_report`, **exactitud balanceada**, **F1-macro** y **recall por clase**. Comentad especialmente el comportamiento en **prediabetes** y **diabetes**.
2. (Multiclase) **ROC-AUC** y/o **PR-AUC** *one-vs-rest* por clase.
3. **Interpretabilidad:** importancia de variables (`permutation_importance`) y/o **SHAP**. ¿Qué factores pesan más (presión, IMC, salud general, edad…)? ¿Es coherente con la literatura médica?
4. **Discusión crítica:** límites del modelo, sesgos posibles, qué mejoraríais.

# <font color="steelblue">Fase 8 — Despliegue del modelo</font>

Un modelo en un cuaderno no sirve a nadie: hay que poder **usarlo**.

**Tareas obligatorias**
1. **Persistencia:** guardad el **`Pipeline` completo** (preprocesado + modelo) con `joblib`, de modo que reciba datos crudos y devuelva la predicción. Comprobad que se puede **recargar y predecir**.
2. **Función de predicción:** `predecir_riesgo(...)` que tome las 21 variables de un individuo y devuelva la clase y las **probabilidades**.
3. **Interfaz interactiva:** una pequeña app con **Gradio** (o `ipywidgets`) donde se introduzcan los valores y se muestre el riesgo estimado. En Colab, Gradio genera un **enlace público** temporal: incluidlo en la entrega.
4. (Opcional, nota extra) Empaquetar como app **Streamlit** o un endpoint **FastAPI**.

> **Aviso clínico:** dejad claro en la interfaz que es una herramienta **educativa**, no un diagnóstico médico.

# <font color="steelblue">Pistas y errores típicos</font>

* **El gran error:** ajustar escalado/SMOTE sobre todo el dataset → **fuga de datos**. Solución: `Pipeline` (imblearn) + CV.
* **No mires solo la *accuracy*.** Un modelo que nunca predice "prediabetes" puede tener *accuracy* alta y ser inútil.
* **Coste computacional:** submuestrea para explorar y reentrena el ganador sobre todo el *train*.
* **Prediabetes (clase 1) es difícil:** es la más rara y la más confundible. Analiza su *recall* con honestidad; quizá convenga discutir un esquema **binario** (diabético / no) como alternativa.
* **Despliegue:** la función de predicción debe recibir las variables en el **mismo orden y formato** que `X`; por eso conviene guardar el **Pipeline entero**, no solo el clasificador.

# <font color="steelblue">Referencias</font>

* Teboul, A. (2022). *Diabetes Health Indicators Dataset*. Kaggle.
* CDC. *Behavioral Risk Factor Surveillance System (BRFSS) 2015*.
* CDC. *CDC Diabetes Health Indicators*. UCI ML Repository (id 891), 2023.
* Cuadernos del curso: *Equilibrando las muestras*, *Boosting*, *Random Forest*, *SVM*, *Regresión logística múltiple*.